# Checkpoint 5A — IGRF / Magnetic-Coordinate Audit + Pilot Framing (NOAA-19, January 2024)

**Question:** what magnetic/IGRF-related information is in the NOAA POES/MetOp files, and can the
Jan-2024 particle-defined high-flux footprint be *described* in magnetic-coordinate terms **without
overclaiming**?

**Conceptual separation (do not collapse):**
- *Particle-defined footprint* = where measured proton flux is high (CP4A/CP4B).
- *Magnetic / IGRF variables* = coordinates / field quantities from a geomagnetic model + ephemeris.

These are **related but not identical**. This checkpoint is an **audit + descriptive pilot** — no
causality, no "the SAA is exactly the field minimum", no final SAA boundary, no dose/health/danger/
discovery claims. The NOAA files already provide IGRF variables, so **no external IGRF package is
added**. `mep_IFC_on==-1` retained, uninterpreted.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from saa.load_poes import open_poes_netcdf
from saa.magnetic_audit import (
    audit_magnetic_variables, build_selection_table, PILOT_MAGNETIC_VARS, INVALID_SENTINELS,
    load_region_with_magnetic, footprint_magnetic_distributions, save_table,
    plot_particle_footprint_reference, plot_flux_vs_variable, plot_high_flux_in_magnetic_space,
    plot_geographic_variable_map,
)
RAW = ROOT/"data"/"raw"; PROC = ROOT/"data"/"processed"; TBL = ROOT/"outputs"/"tables"; FIG = ROOT/"outputs"/"figures"
for d in (PROC, TBL, FIG): d.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 240, "display.max_columns", 30)
SAMPLE_FILE = RAW / "poes_n19_20240101_proc.nc"
print("ROOT:", ROOT)

ROOT: /home/fellipe/Projects/Active/saa-poes-mapping


## 1. Audit magnetic/IGRF-related variables in a real NOAA-19 NetCDF file

In [2]:
with open_poes_netcdf(str(SAMPLE_FILE)) as ds:
    audit = audit_magnetic_variables(ds)
save_table(audit, TBL/"cp5a_magnetic_variable_audit.csv", TBL/"cp5a_magnetic_variable_audit.parquet")
print(f"magnetic/IGRF-related variables found: {len(audit)} (all from real metadata+values)")
print(audit[["variable_name","units","long_name","example_min","example_max","missing_percent","recommended_use"]].to_string(index=False))

magnetic/IGRF-related variables found: 29 (all from real metadata+values)
      variable_name units                                                   long_name   example_min  example_max  missing_percent                    recommended_use
            Bp_foot    nT                              Bphi IGRF (foot of field line) -16370.610352 14944.030273              0.0         caution/component-not-core
             Bp_sat    nT                                       Bphi IGRF (satellite) -10078.809570  9473.910156              0.0         caution/component-not-core
            Br_foot    nT                           Bradial IGRF (foot of field line) -57872.710938 63253.679688              0.0         caution/component-not-core
             Br_sat    nT                                    Bradial IGRF (satellite) -41185.410156 43847.121094              0.0         caution/component-not-core
            Bt_foot    nT                            Btheta IGRF (foot of field line) -35410.578125 1

**Note:** `L_IGRF` uses **-1 as a documented invalid-L sentinel** (see its example_min). It is kept
faithfully in the saved data but excluded from analysis via `INVALID_SENTINELS`.

## 2. Select a small, well-documented set of magnetic variables for the pilot

In [3]:
selection = build_selection_table(audit)
save_table(selection, TBL/"cp5a_selected_magnetic_variables.csv",
           TBL/"cp5a_selected_magnetic_variables.parquet")
print(selection.to_string(index=False))
selected = [v for v in selection.loc[selection.use_in_cp5a, "variable_name"]]
print("\nSELECTED for pilot:", selected)
print("INVALID_SENTINELS applied in analysis:", INVALID_SENTINELS)

      variable_name  use_in_cp5a     use_type                                                                          reason                                                                          caveat
            Bp_foot        False     excluded                                            not selected for the small pilot set                                                                               -
             Bp_sat        False     excluded                                            not selected for the small pilot set                                                                               -
            Br_foot        False     excluded                                            not selected for the small pilot set                                                                               -
             Br_sat        False     excluded                                            not selected for the small pilot set                                                   

## 3. Build the NOAA-19 Jan-2024 regional flux + magnetic dataset (does not overwrite CP4 files)

In [4]:
region, counts, paths = load_region_with_magnetic("2024-01-01", "2024-01-31",
                                                  satellite="noaa19", raw_dir=str(RAW))
out = PROC / "cp5a_noaa19_2024-01_region_flux_plus_magnetic.parquet"
region.to_parquet(out, index=False)
print(f"region rows: {len(region):,} | files: {len(paths)} | counts: {counts}")
print("columns:", list(region.columns))
print("saved", out.name)
# quick faithfulness check vs accepted CP4A region row count
cp4a = pd.read_parquet(PROC / "noaa19_2024-01_mep_omni_flux_p1_region.parquet")
print(f"CP4A region rows: {len(cp4a):,} (CP5A should match: {len(region)==len(cp4a)})")

region rows: 205,153 | files: 31 | counts: {'n_total': 1206770, 'n_after_geo': 205153, 'n_ifc_on_dropped': 0, 'n_ifc_minus1': 192334, 'n_after_ifc': 205153}
columns: ['time', 'lat', 'lon', 'alt', 'satellite', 'source_file', 'mep_omni_flux_p1', 'mep_IFC_on', 'mep_omni_flux_flag_fit', 'L_IGRF', 'Btot_sat', 'mag_lat_sat', 'mag_lon_sat', 'MLT', 'lon180']
saved cp5a_noaa19_2024-01_region_flux_plus_magnetic.parquet
CP4A region rows: 205,153 (CP5A should match: True)


## 4. Inside-vs-outside footprint magnetic distributions (4 pilot cases; descriptive only)

In [5]:
grids = {5: pd.read_parquet(TBL/"cp4a_noaa19_2024-01_grid_5deg.parquet"),
         2: pd.read_parquet(TBL/"cp4a_noaa19_2024-01_grid_2deg.parquet")}
cases = [(5, 90, "top10"), (5, 95, "top5"), (2, 90, "top10"), (2, 95, "top5")]
frames = []; ref_inside = {}; ref_df = {}
for gd, pct, lbl in cases:
    dist, inside_mask, df = footprint_magnetic_distributions(
        region, grids[gd], float(gd), "mean_flux", f"enough_samples_{gd}deg", pct, lbl, grid_deg=gd)
    frames.append(dist)
    ref_inside[(gd,lbl)] = inside_mask; ref_df[(gd,lbl)] = df
dists = pd.concat(frames, ignore_index=True)
save_table(dists, TBL/"cp5a_footprint_magnetic_distributions.csv",
           TBL/"cp5a_footprint_magnetic_distributions.parquet")
print("distribution rows:", len(dists), "(4 cases x", len(PILOT_MAGNETIC_VARS), "vars)")
show = dists[dists.comparison_case=="top10_5deg_mean"]
print(show[["magnetic_variable","count_inside","count_outside","median_inside","median_outside",
            "iqr_inside","iqr_outside"]].to_string(index=False))

distribution rows: 20 (4 cases x 5 vars)
magnetic_variable  count_inside  count_outside  median_inside  median_outside  iqr_inside  iqr_outside
           L_IGRF         20821         180418       1.290000        1.550000    0.130000     1.130000
         Btot_sat         20829         184324   16621.070312    20242.230469  593.019531  3978.401855
      mag_lat_sat         20829         184324      -9.550000      -20.820000   10.520000    49.449998
      mag_lon_sat         20829         184324      18.770000       54.980000   29.750001    60.669996
              MLT         20829         184324       8.900000        9.580000   12.050001    12.199999


## 5. Read the descriptive comparison (narrower magnetic range inside the footprint?)

In [6]:
print("== top10 5deg mean: inside vs outside (median [IQR]) ==")
for _, r in dists[dists.comparison_case=="top10_5deg_mean"].iterrows():
    nar = "NARROWER inside" if (r.iqr_inside < r.iqr_outside) else "not narrower"
    print(f"  {r.magnetic_variable:12s} inside {r.median_inside:10.2f} [IQR {r.iqr_inside:8.2f}] | "
          f"outside {r.median_outside:10.2f} [IQR {r.iqr_outside:8.2f}]  -> {nar}")
print("\nNote: mag_lon_sat wraps [0,360); its IQR is not a meaningful spread (documented caveat).")
print("Btot_sat is the most interpretable: the footprint occupies a LOW, NARROW field-strength band")
print("(descriptive; the SAA is conventionally the field-strength minimum) - NOT asserted as causal.")

== top10 5deg mean: inside vs outside (median [IQR]) ==
  L_IGRF       inside       1.29 [IQR     0.13] | outside       1.55 [IQR     1.13]  -> NARROWER inside
  Btot_sat     inside   16621.07 [IQR   593.02] | outside   20242.23 [IQR  3978.40]  -> NARROWER inside
  mag_lat_sat  inside      -9.55 [IQR    10.52] | outside     -20.82 [IQR    49.45]  -> NARROWER inside
  mag_lon_sat  inside      18.77 [IQR    29.75] | outside      54.98 [IQR    60.67]  -> NARROWER inside
  MLT          inside       8.90 [IQR    12.05] | outside       9.58 [IQR    12.20]  -> NARROWER inside

Note: mag_lon_sat wraps [0,360); its IQR is not a meaningful spread (documented caveat).
Btot_sat is the most interpretable: the footprint occupies a LOW, NARROW field-strength band
(descriptive; the SAA is conventionally the field-strength minimum) - NOT asserted as causal.


## 6. Diagnostic figures (DESCRIPTIVE; no smoothing/interpolation as measurement)

In [7]:
# A. particle footprint geographic reference (top10/top5)
plot_particle_footprint_reference(grids[5], FIG/"cp5a_particle_footprint_geographic_reference.png")
# B-D flux vs magnetic variables
plot_flux_vs_variable(region, "L_IGRF",  FIG/"cp5a_flux_vs_L_IGRF.png", xlabel="L_IGRF (invalid -1 removed)")
plot_flux_vs_variable(region, "mag_lat_sat", FIG/"cp5a_flux_vs_magnetic_latitude.png", xlabel="mag_lat_sat [deg]")
plot_flux_vs_variable(region, "MLT", FIG/"cp5a_flux_vs_MLT.png", xlabel="MLT [hours]")
# extra: flux vs Btot_sat (the most relevant field variable)
plot_flux_vs_variable(region, "Btot_sat", FIG/"cp5a_flux_vs_Btot_sat.png", xlabel="Btot_sat [nT]")
# E. high-flux footprint samples in magnetic-coordinate space (top10 5deg)
plot_high_flux_in_magnetic_space(ref_df[(5,"top10")], ref_inside[(5,"top10")],
                                 FIG/"cp5a_high_flux_samples_magnetic_space.png")
# F. geographic map of Btot_sat (field-strength context)
plot_geographic_variable_map(region, "Btot_sat", FIG/"cp5a_geographic_Btot_sat_map.png", agg="median")
print("CP5A figures:", sorted(p.name for p in FIG.glob("cp5a_*.png")))

CP5A figures: ['cp5a_flux_vs_Btot_sat.png', 'cp5a_flux_vs_L_IGRF.png', 'cp5a_flux_vs_MLT.png', 'cp5a_flux_vs_magnetic_latitude.png', 'cp5a_geographic_Btot_sat_map.png', 'cp5a_high_flux_samples_magnetic_space.png', 'cp5a_particle_footprint_geographic_reference.png']


## 7. Summary

- **Usable magnetic variables:** the NOAA files carry a full IGRF set (29 magnetic/coord variables, 0%
  missing). Selected for the pilot: **L_IGRF, Btot_sat, mag_lat_sat, mag_lon_sat** (descriptive) and
  **MLT** (caution-only). Field components, foot-point and pitch-angle variables were excluded/cautioned
  for a first pilot. `L_IGRF`'s -1 invalid sentinel is excluded in analysis.
- **Footprint occupies a narrower magnetic-coordinate range:** inside the top10/top5 high-flux
  footprint, **Btot_sat** and **L_IGRF** (and **mag_lat_sat**) have markedly smaller IQRs than the full
  regional sample, with the footprint sitting at **low Btot / low L** — i.e. the particle-defined
  footprint corresponds, *descriptively*, to a confined low-field / low-L band. **MLT** does **not**
  discriminate (expected; it is local-time, not a spatial SAA coordinate).
- **Promising for CP5B:** `Btot_sat` and `L_IGRF` (and satellite magnetic latitude). **mag_lon_sat**
  needs wrap-aware handling.
- **What cannot be concluded yet:** nothing causal; the particle footprint is **not** asserted to equal
  the field minimum; no SAA boundary; foot-point vs satellite-coordinate choice not yet resolved; single
  satellite/month/channel.

This is **magnetic-coordinate framing of a particle-defined footprint** — descriptive only, *not* a
final magnetic-field explanation, true SAA boundary, dose, health risk, danger zone, or discovery.